# Assignment 3 — Time Series Forecasting: Household Energy Usage

## Objective

Forecast short-term household energy usage using historical time-based patterns. We will compare three forecasting models:
- **ARIMA** (classical time series)
- **Prophet** (Facebook's library, handles seasonality well)
- **XGBoost** (machine learning approach with lag features)

### Dataset
**Household Power Consumption Dataset** — UCI ML Repository  
Contains household power consumption measured at 1-minute intervals over 4 years.

### Skills Gained
- Time series data parsing and resampling
- Temporal feature engineering
- ARIMA modeling with statsmodels
- Prophet forecasting
- XGBoost with lag features
- Model evaluation (MAE, RMSE)
- Time series visualization

## 1. Setup & Imports

In [1]:
import os
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from io import StringIO
import urllib.request

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

try:
    from prophet import Prophet
except ImportError:
    Prophet = None

from xgboost import XGBRegressor

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

try:
    os.chdir(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)))
except NameError:
    pass

os.makedirs('charts', exist_ok=True)
print('All libraries loaded successfully.')
print(f'Charts will be saved to: {os.path.abspath("charts")}')

Importing plotly failed. Interactive plots will not work.


All libraries loaded successfully.
Charts will be saved to: c:\Users\amjad\Projects\developerhub-data-science-internship-p-2-\Assignment 3\charts


## 2. Data Loading & Preprocessing

In [2]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip'

try:
    import zipfile, io
    with urllib.request.urlopen(url, timeout=20) as response:
        zf = zipfile.ZipFile(io.BytesIO(response.read()))
        df = pd.read_csv(zf.open('household_power_consumption.txt'), sep=';', low_memory=False)
    print('Dataset downloaded from UCI.')
except:
    try:
        df = pd.read_csv('household_power_consumption.txt', sep=';', low_memory=False)
        print('Loaded from local file.')
    except:
        print('Creating synthetic data...')
        dates = pd.date_range('2006-12-16', periods=1000, freq='H')
        np.random.seed(42)
        base = np.sin(np.arange(1000) * 2 * np.pi / 24) * 2 + 3
        df = pd.DataFrame({
            'Date': dates.strftime('%d/%m/%Y'),
            'Time': dates.strftime('%H:%M:%S'),
            'Global_active_power': np.maximum(base + np.random.normal(0, 0.5, 1000), 0)
        })

df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df = df.drop(['Date', 'Time'], axis=1, errors='ignore')
df.set_index('DateTime', inplace=True)
df = df.sort_index()

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')

Dataset downloaded from UCI.
Dataset shape: (2075259, 7)
Date range: 2006-12-16 17:24:00 to 2010-11-26 21:02:00


In [4]:
df = df.replace('?', np.nan)
df = df.astype(float)
df = df.ffill().bfill()

print(f'Missing values: {df.isnull().sum().sum()}')

df_hourly = df.iloc[:, 0].resample('h').mean()  # Change 'H' to 'h'
print(f'Hourly data shape: {df_hourly.shape}')

Missing values: 0
Hourly data shape: (34589,)


## 3. Feature Engineering

In [5]:
data = pd.DataFrame({'power': df_hourly})
data['hour'] = data.index.hour
data['day_of_week'] = data.index.dayofweek
data['month'] = data.index.month
data['is_weekend'] = (data.index.dayofweek >= 5).astype(int)

for lag in [1, 6, 12, 24]:
    data[f'lag_{lag}'] = data['power'].shift(lag)

for window in [6, 12, 24]:
    data[f'rolling_mean_{window}'] = data['power'].rolling(window).mean()

data = data.dropna()
print(f'Data shape with features: {data.shape}')

Data shape with features: (34565, 12)


## 4. Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df_hourly.index, df_hourly.values, linewidth=0.8, color='#2E86C1')
axes[0].set_title('Global Active Power - Full Time Series', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Power (kW)')
axes[0].grid(True, alpha=0.3)

recent = df_hourly.iloc[-30*24:]
axes[1].plot(recent.index, recent.values, linewidth=1.5, color='#E74C3C', marker='o', markersize=3)
axes[1].set_title('Last 30 Days', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Power (kW)')
axes[1].set_xlabel('DateTime')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('charts/01_time_series_overview.png', bbox_inches='tight')
plt.close()
print('Saved: 01_time_series_overview.png')

Saved: 01_time_series_overview.png


In [7]:
hourly_avg = data.groupby('hour')['power'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(hourly_avg.index, hourly_avg.values, color='#3498DB', edgecolor='black')
axes[0].set_title('Average Power by Hour of Day', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Power (kW)')
axes[0].grid(True, alpha=0.3, axis='y')

weekend_avg = data.groupby('is_weekend')['power'].mean()
axes[1].bar(['Weekday', 'Weekend'], weekend_avg.values, color=['#2ECC71', '#E67E22'], edgecolor='black')
axes[1].set_title('Weekday vs Weekend', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Power (kW)')

plt.tight_layout()
plt.savefig('charts/02_temporal_patterns.png', bbox_inches='tight')
plt.close()
print('Saved: 02_temporal_patterns.png')

Saved: 02_temporal_patterns.png


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_acf(df_hourly, lags=48, ax=axes[0], title='ACF')
plot_pacf(df_hourly, lags=48, ax=axes[1], title='PACF')

plt.tight_layout()
plt.savefig('charts/03_acf_pacf.png', bbox_inches='tight')
plt.close()
print('Saved: 03_acf_pacf.png')

Saved: 03_acf_pacf.png


## 5. Train-Test Split

In [9]:
train_size = int(len(data) * 0.9)
train_data = data.iloc[:train_size]
test_data = data.iloc[train_size:]

print(f'Train: {len(train_data)}, Test: {len(test_data)}')
print(f'Test period: {test_data.index.min()} to {test_data.index.max()}')

Train: 31108, Test: 3457
Test period: 2010-07-05 21:00:00 to 2010-11-26 21:00:00


## 6. ARIMA Model

In [10]:
print('Fitting ARIMA(1,1,1)...')
try:
    arima_fit = ARIMA(train_data['power'], order=(1, 1, 1)).fit()
    arima_pred = arima_fit.get_forecast(steps=len(test_data)).predicted_mean
    arima_pred.index = test_data.index
    print('ARIMA fitted.')
except Exception as e:
    print(f'ARIMA error: {e}')
    arima_pred = None

Fitting ARIMA(1,1,1)...
ARIMA fitted.


## 7. Prophet Model

In [12]:
if Prophet is not None:
    print('Fitting Prophet...')
    try:
        pdata = train_data[['power']].reset_index()
        pdata.columns = ['ds', 'y']
        pm = Prophet(yearly_seasonality=False, daily_seasonality=True)
        pm.fit(pdata)
        future = pm.make_future_dataframe(periods=len(test_data), freq='h')
        prophet_pred = pd.Series(pm.predict(future)['yhat'].iloc[-len(test_data):].values, index=test_data.index)
        print('Prophet fitted.')
    except Exception as e:
        print(f'Prophet error: {e}')
        prophet_pred = None
else:
    prophet_pred = None

Fitting Prophet...


06:05:11 - cmdstanpy - INFO - Chain [1] start processing
06:05:28 - cmdstanpy - INFO - Chain [1] done processing


Prophet fitted.


## 8. XGBoost Model

In [13]:
print('Fitting XGBoost...')
try:
    feat_cols = [c for c in data.columns if c != 'power']
    X_train = train_data[feat_cols]
    y_train = train_data['power']
    X_test = test_data[feat_cols]
    
    xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbose=0)
    xgb.fit(X_train, y_train, verbose=False)
    xgb_pred = pd.Series(xgb.predict(X_test), index=test_data.index)
    print('XGBoost fitted.')
except Exception as e:
    print(f'XGBoost error: {e}')
    xgb_pred = None

Fitting XGBoost...
XGBoost fitted.


## 9. Model Evaluation

In [14]:
y_true = test_data['power'].values
results = {}

for name, pred in [('ARIMA', arima_pred), ('Prophet', prophet_pred), ('XGBoost', xgb_pred)]:
    if pred is not None:
        mae = mean_absolute_error(y_true, pred.values)
        rmse = np.sqrt(mean_squared_error(y_true, pred.values))
        mape = np.mean(np.abs((y_true - pred.values) / y_true)) * 100
        results[name] = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
        print(f'{name:12} → MAE: {mae:.4f} | RMSE: {rmse:.4f} | MAPE: {mape:.2f}%')

results_df = pd.DataFrame(results).T
print('\n=== Model Comparison ===')
print(results_df)

ARIMA        → MAE: 0.5688 | RMSE: 0.7164 | MAPE: 104.27%
Prophet      → MAE: 0.5303 | RMSE: 0.6985 | MAPE: 99.75%
XGBoost      → MAE: 0.2839 | RMSE: 0.4145 | MAPE: 39.88%

=== Model Comparison ===
              MAE      RMSE        MAPE
ARIMA    0.568815  0.716378  104.269228
Prophet  0.530262  0.698476   99.752094
XGBoost  0.283883  0.414531   39.878576


In [15]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, metric in enumerate(['MAE', 'RMSE', 'MAPE']):
    vals = results_df[metric].values
    axes[i].bar(range(len(vals)), vals, color=['#3498DB', '#2ECC71', '#E74C3C'][:len(vals)], edgecolor='black')
    axes[i].set_xticks(range(len(vals)))
    axes[i].set_xticklabels(results_df.index)
    axes[i].set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    axes[i].set_ylabel(metric)
    for j, v in enumerate(vals):
        axes[i].text(j, v + 0.01*v, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Model Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/04_model_comparison.png', bbox_inches='tight')
plt.close()
print('Saved: 04_model_comparison.png')

Saved: 04_model_comparison.png


## 10. Forecast Visualization

In [16]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for idx, (name, pred) in enumerate([('ARIMA', arima_pred), ('Prophet', prophet_pred), ('XGBoost', xgb_pred)]):
    if pred is not None:
        axes[idx].plot(test_data.index, y_true, label='Actual', linewidth=2, color='black')
        axes[idx].plot(test_data.index, pred.values, label='Forecast', linewidth=1.5,
                       color=['#3498DB', '#2ECC71', '#E74C3C'][idx], alpha=0.8)
        axes[idx].fill_between(test_data.index, y_true, pred.values, alpha=0.2)
        mae = results[name]['MAE']
        axes[idx].set_title(f'{name} — MAE: {mae:.4f}', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Power (kW)')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('DateTime')
plt.suptitle('Actual vs Forecasted', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/05_forecast_full.png', bbox_inches='tight')
plt.close()
print('Saved: 05_forecast_full.png')

Saved: 05_forecast_full.png


In [17]:
zoom_idx = 7*24
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for idx, (name, pred) in enumerate([('ARIMA', arima_pred), ('Prophet', prophet_pred), ('XGBoost', xgb_pred)]):
    if pred is not None:
        axes[idx].plot(test_data.index[:zoom_idx], y_true[:zoom_idx], label='Actual', linewidth=2.5, color='black', marker='o', markersize=3)
        axes[idx].plot(test_data.index[:zoom_idx], pred.values[:zoom_idx], label='Forecast', linewidth=2,
                       color=['#3498DB', '#2ECC71', '#E74C3C'][idx], alpha=0.8, marker='s', markersize=3)
        axes[idx].fill_between(test_data.index[:zoom_idx], y_true[:zoom_idx], pred.values[:zoom_idx], alpha=0.15)
        axes[idx].set_title(f'{name} — First 7 Days', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Power (kW)')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('DateTime')
plt.suptitle('Zoomed: First 7 Days', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/06_forecast_zoomed.png', bbox_inches='tight')
plt.close()
print('Saved: 06_forecast_zoomed.png')

Saved: 06_forecast_zoomed.png


In [18]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for idx, (name, pred) in enumerate([('ARIMA', arima_pred), ('Prophet', prophet_pred), ('XGBoost', xgb_pred)]):
    if pred is not None:
        res = y_true - pred.values
        axes[idx].plot(test_data.index, res, linewidth=1, color=['#3498DB', '#2ECC71', '#E74C3C'][idx], alpha=0.7)
        axes[idx].axhline(y=0, color='black', linestyle='--', linewidth=1.5)
        axes[idx].fill_between(test_data.index, res, 0, alpha=0.3)
        axes[idx].set_title(f'{name} — Residuals', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Residual (kW)')
        axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('DateTime')
plt.suptitle('Residuals Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/07_residuals.png', bbox_inches='tight')
plt.close()
print('Saved: 07_residuals.png')

Saved: 07_residuals.png


## 11. Feature Importance

In [19]:
if xgb_pred is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    fi = xgb.feature_importances_
    idx = np.argsort(fi)[-10:]
    ax.barh(range(len(idx)), fi[idx], color='#3498DB', edgecolor='black')
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels(feat_cols[i] for i in idx)
    ax.set_xlabel('Importance')
    ax.set_title('XGBoost — Top 10 Features', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('charts/08_feature_importance.png', bbox_inches='tight')
    plt.close()
    print('Saved: 08_feature_importance.png')

Saved: 08_feature_importance.png


## 12. Summary

In [20]:
print('='*70)
print('TIME SERIES FORECASTING — HOUSEHOLD ENERGY')
print('='*70)
print(f'Dataset: {len(df_hourly):,} hourly records')
print(f'Avg power: {df_hourly.mean():.2f} kW | Range: {df_hourly.min():.2f} - {df_hourly.max():.2f} kW')
print()
print('MODEL RESULTS:')
for name in results_df.index:
    print(f'  {name:12} → MAE: {results_df.loc[name, "MAE"]:8.4f} | RMSE: {results_df.loc[name, "RMSE"]:8.4f} | MAPE: {results_df.loc[name, "MAPE"]:6.2f}%')
print()
best = results_df['MAE'].idxmin()
print(f'Best model: {best}')
print('='*70)

TIME SERIES FORECASTING — HOUSEHOLD ENERGY
Dataset: 34,589 hourly records
Avg power: 1.09 kW | Range: 0.12 - 6.56 kW

MODEL RESULTS:
  ARIMA        → MAE:   0.5688 | RMSE:   0.7164 | MAPE: 104.27%
  Prophet      → MAE:   0.5303 | RMSE:   0.6985 | MAPE:  99.75%
  XGBoost      → MAE:   0.2839 | RMSE:   0.4145 | MAPE:  39.88%

Best model: XGBoost
